# E17 — O que a margem não conta

O andar mediu o mundo que faltou e o passado apagado. Este fecha o terceiro pedaço, que junta duas
coisas que a espiral já encontrou separadas: o vigia de margem não vê o que acontece **entre** as
pernas, e ler uma série pelo segundo momento é ler metade dela.

O experimento é controlado pelo segundo momento. Um par tem, por construção, sempre a mesma
correlação declarada; o que muda de um mundo para o outro é a forma como os dias ruins chegam
juntos: com probabilidade p as duas pernas recebem o mesmo choque, de tamanho f, e a correlação do
corpo é resolvida para que a correlação total continue a mesma.

Se o segundo momento resumisse o risco conjunto, nada mudaria entre esses mundos. A pergunta do
caderno é quanto muda.


In [1]:
# <- brinque com: RHO, FAMILIA, DIAS, SEMENTES, JANELA, CAUDA
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import frevolab
from frevolab import dependencia, graficos, mudanca

RHO = 0.5
FAMILIA = (("controle", 0.0, 0.0), ("dois por cento", 0.02, 3.0), ("cinco por cento", 0.05, 3.0),
           ("dez por cento", 0.10, 3.0), ("choque grande", 0.05, 5.0))
DIAS = 8000
SEMENTES = 20
JANELA = 252
CAUDA = 0.05
SIGMA = 0.01
SEMENTE = 700
PESO = 0.5

print("correlacao declarada %.2f | %d dias | %d sementes | corte no pior %.0f%%"
      % (RHO, DIAS, SEMENTES, 100 * CAUDA))


correlacao declarada 0.50 | 8000 dias | 20 sementes | corte no pior 5%


In [2]:
# A familia: mesma correlacao, caudas diferentes.
linhas = []
for nome, p, f in FAMILIA:
    cors, curtoses, excessos, perdas, alem = [], [], [], [], []
    for s in range(SEMENTES):
        a, b = mudanca.par_de_cauda(DIAS, np.random.default_rng(SEMENTE + s), SIGMA, RHO, p, f)
        cors.append(float(np.corrcoef(a, b)[0, 1]))
        padrao = (a - a.mean()) / a.std(ddof=1)
        curtoses.append(float((padrao ** 4).mean()))
        indice = pd.RangeIndex(DIAS)
        ra = dependencia.rompimentos(pd.Series(a, index=indice), JANELA, CAUDA)
        rb = dependencia.rompimentos(pd.Series(b, index=indice), JANELA, CAUDA)
        excessos.append(float(dependencia.juntos(ra, rb)["excesso"]))
        carteira = PESO * a + (1 - PESO) * b
        corte = np.quantile(carteira, CAUDA)
        perdas.append(float(np.mean(carteira[carteira <= corte])))
        pior_a = np.quantile(a, CAUDA)
        alem.append(float(np.mean(a[a <= pior_a] - pior_a)))
    linhas.append({"mundo": nome, "p": p, "f": f, "correlacao": float(np.mean(cors)),
                   "curtose": float(np.mean(curtoses)), "excesso": float(np.mean(excessos)),
                   "perda": float(np.mean(perdas)), "alem_do_corte": float(np.mean(alem))})
familia = pd.DataFrame(linhas).set_index("mundo")
print(familia.round(4).to_string())
print()
print("a correlacao vai de %.3f a %.3f e a perda conjunta de %.5f a %.5f"
      % (familia["correlacao"].min(), familia["correlacao"].max(), familia["perda"].max(),
         familia["perda"].min()))


                    p    f  correlacao  curtose  excesso   perda  alem_do_corte
mundo                                                                          
controle         0.00  0.0      0.5006   2.9915   5.0129 -0.0180        -0.0041
dois por cento   0.02  3.0      0.5000   5.7680   5.5720 -0.0198        -0.0057
cinco por cento  0.05  3.0      0.4977   7.4907   6.8961 -0.0225        -0.0078
dez por cento    0.10  3.0      0.4941   8.1920   9.7963 -0.0273        -0.0110
choque grande    0.05  5.0      0.4947  19.5621   7.0261 -0.0278        -0.0143

a correlacao vai de 0.494 a 0.501 e a perda conjunta de -0.01795 a -0.02784


In [3]:
# A margem: o que ela promete é mantido, o que ela paga não.
linhas = []
for nome, p, f in FAMILIA:
    taxas, profundidades = [], []
    for s in range(SEMENTES):
        a, b = mudanca.par_de_cauda(DIAS, np.random.default_rng(SEMENTE + s), SIGMA, RHO, p, f)
        indice = pd.RangeIndex(DIAS)
        ra = dependencia.rompimentos(pd.Series(a, index=indice), JANELA, CAUDA)
        taxas.append(float(ra.mean()))
        corte = np.quantile(a, CAUDA)
        profundidades.append(float(np.mean(a[a <= corte] - corte)))
    linhas.append({"mundo": nome, "taxa_de_rompimento": float(np.mean(taxas)),
                   "profundidade_media": float(np.mean(profundidades))})
margem = pd.DataFrame(linhas).set_index("mundo")
print(margem.round(4).to_string())
print()
print("a taxa de rompimento fica em %.3f em todos os mundos: o corte e um quantil, e quantil nao erra a taxa"
      % margem["taxa_de_rompimento"].mean())


                 taxa_de_rompimento  profundidade_media
mundo                                                  
controle                     0.0512             -0.0041
dois por cento               0.0512             -0.0057
cinco por cento              0.0508             -0.0078
dez por cento                0.0514             -0.0110
choque grande                0.0509             -0.0143

a taxa de rompimento fica em 0.051 em todos os mundos: o corte e um quantil, e quantil nao erra a taxa


In [4]:
# Figura 1: a correlacao parada e a perda conjunta subindo.
fig, esq = plt.subplots(figsize=(8.6, 4.4))
dir_ = esq.twinx()
posicoes = np.arange(len(FAMILIA))
esq.plot(posicoes, familia["correlacao"], marker="o", color="#1f4e79", lw=1.8,
         label="correlacao (eixo da esquerda)")
dir_.plot(posicoes, -familia["perda"], marker="s", color="#b03a2e", lw=1.8,
          label="perda conjunta no pior 5% (eixo da direita)")
esq.set_ylim(0.0, 1.0)
esq.set_xticks(posicoes)
esq.set_xticklabels([n for n, _, _ in FAMILIA], fontsize=8, rotation=15)
esq.set_ylabel("correlacao medida", color="#1f4e79")
dir_.set_ylabel("perda media no pior 5%", color="#b03a2e")
esq.grid(alpha=0.25, axis="y")
esq.legend(frameon=False, fontsize=8, loc="upper left")
dir_.legend(frameon=False, fontsize=8, loc="lower right")
fig.tight_layout()
graficos.salvar(fig, "E17_o_que_a_margem_nao_conta", 1)
plt.close(fig)
print("correlacoes: %s" % [round(v, 3) for v in familia["correlacao"]])
print("perdas: %s" % [round(v, 5) for v in familia["perda"]])


correlacoes: [0.501, 0.5, 0.498, 0.494, 0.495]
perdas: [-0.01795, -0.0198, -0.02249, -0.02735, -0.02784]


In [5]:
# Figura 2: a nuvem das duas pernas, no mundo gaussiano e no mundo de choque.
fig, (esq, dir_) = plt.subplots(1, 2, figsize=(9.8, 4.6), sharex=True, sharey=True)
for eixo, (nome, p, f) in zip((esq, dir_), (FAMILIA[0], FAMILIA[2])):
    a, b = mudanca.par_de_cauda(4000, np.random.default_rng(99), SIGMA, RHO, p, f)
    eixo.scatter(a, b, s=4, color="#1f4e79" if p == 0.0 else "#b03a2e", alpha=0.5)
    eixo.set_title("%s (correlacao %.2f)" % (nome, float(np.corrcoef(a, b)[0, 1])), fontsize=10)
    eixo.set_xlabel("perna A")
    eixo.grid(alpha=0.25)
esq.set_ylabel("perna B")
fig.tight_layout()
graficos.salvar(fig, "E17_o_que_a_margem_nao_conta", 2)
plt.close(fig)
print("a nuvem do choque tem braços que a gaussiana nao tem")


a nuvem do choque tem braços que a gaussiana nao tem


## Leitura visual das figuras

A preencher olhando os .png com a ponte de visão (AGENTS.md §9). Observação, não número.


In [6]:
# O resultado: um objeto por grandeza, para o livro citar por comando.
NOMES = {"controle": "controle", "dois por cento": "dois_por_cento", "cinco por cento": "cinco_por_cento",
         "dez por cento": "dez_por_cento", "choque grande": "choque_grande"}
resultado = {
    "margem_rho": float(RHO),
    "margem_dias": int(DIAS),
    "margem_sementes": int(SEMENTES),
    "margem_janela": int(JANELA),
    "margem_cauda": float(CAUDA),
    "margem_peso": float(PESO),
    "margem_mundos": int(len(FAMILIA)),
    "margem_correlacao_menor": float(familia["correlacao"].min()),
    "margem_correlacao_maior": float(familia["correlacao"].max()),
    "margem_perda_menor": float(familia["perda"].max()),
    "margem_perda_maior": float(familia["perda"].min()),
    "margem_perda_razao": float(familia["perda"].min() / familia["perda"].max()),
    "margem_excesso_menor": float(familia["excesso"].min()),
    "margem_excesso_maior": float(familia["excesso"].max()),
    "margem_curtose_menor": float(familia["curtose"].min()),
    "margem_curtose_maior": float(familia["curtose"].max()),
    "margem_taxa_de_rompimento": float(margem["taxa_de_rompimento"].mean()),
}
for nome, curto in NOMES.items():
    resultado["margem_correlacao_%s" % curto] = float(familia.loc[nome, "correlacao"])
    resultado["margem_curtose_%s" % curto] = float(familia.loc[nome, "curtose"])
    resultado["margem_excesso_%s" % curto] = float(familia.loc[nome, "excesso"])
    resultado["margem_perda_%s" % curto] = float(familia.loc[nome, "perda"])
    resultado["margem_profundidade_%s" % curto] = float(margem.loc[nome, "profundidade_media"])

caminho = Path("lab/resultados/E17_o_que_a_margem_nao_conta.json")
caminho.write_text(json.dumps(resultado, indent=1, ensure_ascii=False, sort_keys=True), encoding="utf-8")
print("%s gravado | %d grandezas" % (caminho, len(resultado)))


lab/resultados/E17_o_que_a_margem_nao_conta.json gravado | 42 grandezas
